# Imports and Setup

In [10]:
import os
from IPython.display import display
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from imblearn.pipeline import Pipeline
from imblearn.over_sampling import SMOTE

from sklearn.utils.class_weight import compute_class_weight
from sklearn.utils.class_weight import compute_sample_weight
from sklearn.model_selection import StratifiedKFold, GridSearchCV

from sklearn.neighbors import KNeighborsClassifier

from sklearn.metrics import roc_auc_score, accuracy_score, confusion_matrix, recall_score, precision_score, f1_score, balanced_accuracy_score, classification_report
from sklearn.metrics import classification_report, confusion_matrix

import warnings
warnings.filterwarnings('ignore')

In [6]:
# Setting up the root directory
root_dir = r"C:\Users\Source\OneDrive - IESEG\Desktop\MBD\ML\Kaggle_Competition"

os.chdir(root_dir)

os.getcwd()

'C:\\Users\\Source\\OneDrive - IESEG\\Desktop\\MBD\\ML\\Kaggle_Competition'

# II - Load Data

### Loading the cleaned data

In [7]:
train = pd.read_csv(os.path.join(root_dir, 'data', 'processed', 'train_preprocessed.csv'))
train_labels = pd.read_csv(os.path.join(root_dir, 'data', 'processed', 'y_train.csv'))
test = pd.read_csv(os.path.join(root_dir, 'data', 'processed', 'test_preprocessed.csv'))

In [8]:
# Verifying the imports
display(train.head(3))
display(train_labels.head(3))
display(test.head(3))

,Var1,Var2,Var3,Var4,Var5,Var6,Var7,Var8,Var9,Var10,...,Var14822_freq,Var14893_freq,Var14904_freq,Var14910_freq,Var14913_freq,Var14923_freq,Var14965_freq,Var14970_freq,Var14990_freq,Var14993_freq
0,0.0,0.0,0.0,0.0,6.0,0.0,0.0,0.0,0.0,0.0,...,0.000167,0.000833,0.000033,0.959833,0.000233,0.5867,0.071467,0.079633,0.082333,0.000833
1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.000067,0.001433,0.007033,0.959833,0.000600,0.5867,0.005167,0.347167,0.082333,0.001433
2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.000033,0.000167,0.002667,0.959833,0.000100,0.5867,0.002233,0.347167,0.028400,0.000167


,Target_appetency
0,0
1,0
2,0


,Var1,Var2,Var3,Var4,Var5,Var6,Var7,Var8,Var9,Var10,...,Var14822_freq,Var14893_freq,Var14904_freq,Var14910_freq,Var14913_freq,Var14923_freq,Var14965_freq,Var14970_freq,Var14990_freq,Var14993_freq
0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.000667,0.000600,0.001300,0.959833,0.000700,0.128267,0.089000,0.057333,0.162700,0.000600
1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.001333,0.001767,0.001100,0.959833,0.001367,0.128267,0.009433,0.023033,0.022400,0.001767
2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.000000,0.000400,0.011567,0.959833,0.000100,0.128267,0.006467,0.014467,0.027333,0.000400


# III - KNN Classifier

In [ ]:
X_train = train.copy()
y_train = train_labels['Target_appetency'].values

# We define our pipeline
# We use imblearn pipeline instead of sklear pipeline as we will use SMOTE during fit and not transform, which is not supported by sklearn
pipeline = Pipeline([
    ('scaler', StandardScaler()), # We scale inside out CV to avoid leakage (and not in preprocessing !)
    ('pca', PCA(random_state=42)), # We 
    ('smote', SMOTE(random_state=42)),
    ('knn', KNeighborsClassifier())
])

# Defining the tuning parameters
param_grid = {
    'pca__n_components' : [30,50,100],
    'knn__n_neighbors' : [3, 5, 11, 21, 51],
    'knn__metric' : ['euclidean', 'manhattan', 'minkowski'],
    'knn__weights' : ['uniform', 'distance']
}

# K-fold with stratified (to preserve the minority class ratio)
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# Grid seach with scoring using the AUC
grid_search = GridSearchCV(
    estimator=pipeline,
    param_grid=param_grid,
    scoring='roc_auc',
    cv=cv,
    n_jobs=1,
    verbose=2,
    refit=True  # refit best model on full train at the end
)

# fitting
grid_search.fit(X_train, y_train)

Fitting 5 folds for each of 90 candidates, totalling 450 fits
[CV] END knn__metric=euclidean, knn__n_neighbors=3, knn__weights=uniform, pca__n_components=30; total time=  20.9s
[CV] END knn__metric=euclidean, knn__n_neighbors=3, knn__weights=uniform, pca__n_components=30; total time=  20.1s
[CV] END knn__metric=euclidean, knn__n_neighbors=3, knn__weights=uniform, pca__n_components=30; total time=  17.0s
[CV] END knn__metric=euclidean, knn__n_neighbors=3, knn__weights=uniform, pca__n_components=30; total time=  17.5s
[CV] END knn__metric=euclidean, knn__n_neighbors=3, knn__weights=uniform, pca__n_components=30; total time=  16.6s
[CV] END knn__metric=euclidean, knn__n_neighbors=3, knn__weights=uniform, pca__n_components=50; total time=  19.1s
[CV] END knn__metric=euclidean, knn__n_neighbors=3, knn__weights=uniform, pca__n_components=50; total time=  21.7s
[CV] END knn__metric=euclidean, knn__n_neighbors=3, knn__weights=uniform, pca__n_components=50; total time=  20.5s
[CV] END knn__metr

In [12]:
# Printing the results 
print(f'Best AUC (CV)  : {grid_search.best_score_}')
print(f'Best params  : {grid_search.best_params_}')

AttributeError: 'GridSearchCV' object has no attribute 'best_score_'